In [1]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/katabatic1


Mounted at /content/drive
/content/drive/MyDrive/katabatic1


In [2]:
from katabatic.models.ctabgan_plus.models import CTABGANPlus
print("CTAB-GAN+ import successful")


CTAB-GAN+ import successful


In [ ]:
from pathlib import Path
import pandas as pd
import time
import shutil
import numpy as np

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "adult.csv"
DATASET_DIR = ROOT / "sample_data" / "adult"
SYNTH_DIR = ROOT / "synthetic" / "adult" / "ctabgan_plus"

DATASET_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
pipeline_start = time.time()

from katabatic.utils.split_dataset import split_dataset

print("Splitting ADULT dataset")

split_dataset(
    input_csv=str(RAW_CSV),
    output_dir=str(DATASET_DIR),
    label_col="income",
    test_size=0.2,
    stratify=True,
    random_state=42
)

print("Split complete")

from katabatic.models.ctabgan_plus.adapter import CTABGANPlusAdapter

print("Training CTAB-GAN+ (ADULT, 300 epochs)")

train_start = time.time()

model = CTABGANPlusAdapter(
    config={"epochs": 300}
)

model.train(
    dataset_dir=str(DATASET_DIR),
    synthetic_dir=str(SYNTH_DIR)
)

train_end = time.time()
print(f"Training complete in {(train_end - train_start)/60:.2f} minutes")

x_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")
y_synth = pd.read_csv(SYNTH_DIR / "y_synth.csv")

print("Synthetic shapes:")
print("X:", x_synth.shape)
print("y:", y_synth.shape)

print("Synthetic label distribution:")
print(y_synth.value_counts(normalize=True))

from sklearn.preprocessing import OrdinalEncoder

print("Encoding features for TSTR (unknown-safe)")

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

x_test = pd.read_csv(DATASET_DIR / "x_test.csv")
x_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")

x_test_enc = encoder.fit_transform(x_test.astype(str))
x_synth_enc = encoder.transform(x_synth.astype(str))

pd.DataFrame(x_test_enc, columns=x_test.columns).to_csv(
    DATASET_DIR / "x_test.csv", index=False
)

pd.DataFrame(x_synth_enc, columns=x_synth.columns).to_csv(
    SYNTH_DIR / "x_synth.csv", index=False
)

print("Feature encoding complete")

print("Reindexing labels for XGBoost compatibility")

y_train = pd.read_csv(DATASET_DIR / "y_train.csv")
y_test = pd.read_csv(DATASET_DIR / "y_test.csv")
y_synth = pd.read_csv(SYNTH_DIR / "y_synth.csv")

label_col = y_train.columns[0]

all_labels = pd.concat([y_train, y_test, y_synth])[label_col].unique()
all_labels = sorted(all_labels)

label_map = {old: new for new, old in enumerate(all_labels)}
print("Label mapping:", label_map)

for df in [y_train, y_test, y_synth]:
    df[label_col] = df[label_col].map(label_map)

y_train.to_csv(DATASET_DIR / "y_train.csv", index=False)
y_test.to_csv(DATASET_DIR / "y_test.csv", index=False)
y_synth.to_csv(SYNTH_DIR / "y_synth.csv", index=False)

print("Label reindexing complete")

from katabatic.evaluate.tstr.evaluation import TSTREvaluation

print("Running TSTR evaluation")

tstr_start = time.time()

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(DATASET_DIR)
)

results = tstr.evaluate()

tstr_end = time.time()
print(f"TSTR completed in {(tstr_end - tstr_start)/60:.2f} minutes")

print("TSTR Results")
print(results)

pipeline_end = time.time()
print(f"Total pipeline runtime: {(pipeline_end - pipeline_start)/60:.2f} minutes")

print("CTAB-GAN+ (ADULT, 300 EPOCHS) PIPELINE FINISHED SUCCESSFULLY")


ROOT: /content/drive/MyDrive/katabatic1

▶ Splitting ADULT dataset
Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
<=50K    0.759175
>50K     0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
<=50K    0.759251
>50K     0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
✔ Split complete

▶ Training CTAB-GAN+ (ADULT, 300 epochs)


/usr/local/lib/python3.12/dist-packages/sklearn/mixture/_base.py:269: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/mixture/_base.py:269: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/mixture/_base.py:269: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/mixture/_base.py:269: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(
/usr/local/lib/python3.1

✔ Training complete in 8.71 minutes

Synthetic shapes:
X: (19536, 14)
y: (19536, 1)

Synthetic label distribution:
class
<=50K    0.654894
>50K     0.345106
Name: proportion, dtype: float64

▶ Encoding features for TSTR (unknown-safe)
✔ Feature encoding complete

▶ Reindexing labels for XGBoost compatibility
Label mapping: {' <=50K': 0, ' >50K': 1}
✔ Label reindexing complete

▶ Running TSTR evaluation


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



Results saved to: Results/adult/ctabgan_plus_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7060
F1 Score: 0.7182
AUC: 0.6952

MLP:
Accuracy: 0.7565
F1 Score: 0.7668
AUC: 0.8039

RF:
Accuracy: 0.7758
F1 Score: 0.7779
AUC: 0.7978

XGBoost:
Accuracy: 0.7821
F1 Score: 0.7879
AUC: 0.8283
✔ TSTR completed in 0.27 minutes

📊 TSTR RESULTS
{'LR': {'Accuracy': 0.7059726700445264, 'F1 Score': 0.7182316315242225, 'AUC': np.float64(0.695245596974887)}, 'MLP': {'Accuracy': 0.7564870259481038, 'F1 Score': 0.7668076589203863, 'AUC': np.float64(0.8039267400590165)}, 'RF': {'Accuracy': 0.7758329494856441, 'F1 Score': 0.7778860423422088, 'AUC': np.float64(0.7977625049008481)}, 'XGBoost': {'Accuracy': 0.7821280515891295, 'F1 Score': 0.7878656169865548, 'AUC': np.float64(0.8283492782856318)}}

⏱️ Total pipeline runtime: 9.13 minutes

✅ CTAB-GAN+ (ADULT, 300 EPOCHS) PIPELINE FINISHED SUCCESSFULLY


In [ ]:
from pathlib import Path
import pandas as pd
import time
import shutil

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "nursery.csv"
DATASET_DIR = ROOT / "sample_data" / "nursery"
SYNTH_DIR = ROOT / "synthetic" / "nursery" / "ctabgan_plus"

DATASET_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
pipeline_start = time.time()

from katabatic.utils.split_dataset import split_dataset

print("Splitting NURSERY dataset")

split_dataset(
    input_csv=str(RAW_CSV),
    output_dir=str(DATASET_DIR),
    label_col="target",
    test_size=0.2,
    stratify=True,
    random_state=42
)

print("Split complete")

from katabatic.models.ctabgan_plus.adapter import CTABGANPlusAdapter

print("Training CTAB-GAN+ (NURSERY — categorical only)")

train_start = time.time()

model = CTABGANPlusAdapter()
model.train(
    dataset_dir=str(DATASET_DIR),
    synthetic_dir=str(SYNTH_DIR)
)

train_end = time.time()
print(f"Training completed in {(train_end - train_start)/60:.2f} minutes")

x_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")
y_synth = pd.read_csv(SYNTH_DIR / "y_synth.csv")

print("Synthetic shapes:")
print("X:", x_synth.shape)
print("y:", y_synth.shape)

print("Synthetic label distribution:")
print(y_synth.value_counts(normalize=True))

from sklearn.preprocessing import OrdinalEncoder

print("Encoding for TSTR (unknown-safe)")

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

x_test = pd.read_csv(DATASET_DIR / "x_test.csv")
x_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")

x_test_enc = encoder.fit_transform(x_test.astype(str))
x_synth_enc = encoder.transform(x_synth.astype(str))

pd.DataFrame(x_test_enc, columns=x_test.columns).to_csv(
    DATASET_DIR / "x_test.csv", index=False
)
pd.DataFrame(x_synth_enc, columns=x_synth.columns).to_csv(
    SYNTH_DIR / "x_synth.csv", index=False
)

print("Encoding complete")

from katabatic.evaluate.tstr.evaluation import TSTREvaluation

print("Running TSTR evaluation")

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(DATASET_DIR)
)

results = tstr.evaluate()

print("TSTR Results")
print(results)

print(f"Total runtime: {(time.time() - pipeline_start)/60:.2f} minutes")
print("NURSERY PIPELINE FINISHED (ERROR-FREE)")


ROOT: /content/drive/MyDrive/katabatic1

▶ Splitting NURSERY dataset
Loaded data with shape: (12960, 9)
Saved train/test full data
Train size: (10368, 9), Test size: (2592, 9)
Train label distribution:
 8
not_recom     0.333333
priority      0.329186
spec_prior    0.312018
very_recom    0.025270
recommend     0.000193
Name: proportion, dtype: float64
Test label distribution:
 8
not_recom     0.333333
priority      0.329090
spec_prior    0.312114
very_recom    0.025463
Name: proportion, dtype: float64
Saved X/y split
Training shape: (10368, 8) (10368,)
Test shape: (2592, 8) (2592,)
✔ Split complete

▶ Training CTAB-GAN+ (NURSERY — categorical only)
✔ Training completed in 2.01 minutes

Synthetic shapes:
X: (7776, 8)
y: (7776, 1)

Synthetic label distribution:
8         
not_recom     0.347094
priority      0.342207
spec_prior    0.306713
very_recom    0.003987
Name: proportion, dtype: float64

▶ Encoding for TSTR (unknown-safe)
✔ Encoding complete

▶ Running TSTR evaluation


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [11:54:25] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/nursery/ctabgan_plus_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7593
F1 Score: 0.7509

MLP:
Accuracy: 0.8835
F1 Score: 0.8729

RF:
Accuracy: 0.8353
F1 Score: 0.8259

XGBoost:
Accuracy: 0.8596
F1 Score: 0.8502

📊 TSTR RESULTS
{'LR': {'Accuracy': 0.7592592592592593, 'F1 Score': 0.7508691529709229}, 'MLP': {'Accuracy': 0.8834876543209876, 'F1 Score': 0.8729394812641018}, 'RF': {'Accuracy': 0.8352623456790124, 'F1 Score': 0.8258871359733828}, 'XGBoost': {'Accuracy': 0.8595679012345679, 'F1 Score': 0.8501768466340371}}

⏱️ Total runtime: 2.16 minutes

✅ NURSERY PIPELINE FINISHED (ERROR-FREE)


In [ ]:
from pathlib import Path
import pandas as pd
import shutil
import time

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "magic.csv"
DATASET_DIR = ROOT / "sample_data" / "magic"
SYNTH_DIR = ROOT / "synthetic" / "magic" / "ctabgan_plus"

DATASET_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
pipeline_start = time.time()

from katabatic.utils.split_dataset import split_dataset

print("Splitting MAGIC dataset")

split_dataset(
    input_csv=str(RAW_CSV),
    output_dir=str(DATASET_DIR),
    label_col="class",
    test_size=0.2,
    stratify=True,
    random_state=42
)

print("Split complete")

from katabatic.models.ctabgan_plus.adapter import CTABGANPlusAdapter

print("Training CTAB-GAN+ (MAGIC)")

train_start = time.time()

model = CTABGANPlusAdapter()
model.train(
    dataset_dir=str(DATASET_DIR),
    synthetic_dir=str(SYNTH_DIR)
)

train_end = time.time()
print(f"Training complete in {(train_end - train_start)/60:.2f} minutes")

x_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")
y_synth = pd.read_csv(SYNTH_DIR / "y_synth.csv")

print("Synthetic shapes:")
print("X:", x_synth.shape)
print("y:", y_synth.shape)

print("Synthetic label distribution:")
print(y_synth.value_counts(normalize=True))

from sklearn.preprocessing import OrdinalEncoder

print("Encoding data for TSTR")

x_test = pd.read_csv(DATASET_DIR / "x_test.csv")
x_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")

encoder = OrdinalEncoder()

x_test_enc = encoder.fit_transform(x_test.astype(str))
x_synth_enc = encoder.transform(x_synth.astype(str))

pd.DataFrame(x_test_enc, columns=x_test.columns).to_csv(
    DATASET_DIR / "x_test_enc.csv", index=False
)
pd.DataFrame(x_synth_enc, columns=x_synth.columns).to_csv(
    SYNTH_DIR / "x_synth_enc.csv", index=False
)

shutil.move(DATASET_DIR / "x_test_enc.csv", DATASET_DIR / "x_test.csv")
shutil.move(SYNTH_DIR / "x_synth_enc.csv", SYNTH_DIR / "x_synth.csv")

print("Encoding complete")

from katabatic.evaluate.tstr.evaluation import TSTREvaluation

print("Running TSTR evaluation")

tstr_start = time.time()

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(DATASET_DIR)
)

results = tstr.evaluate()

tstr_end = time.time()
print(f"TSTR completed in {(tstr_end - tstr_start)/60:.2f} minutes")

print("TSTR Results")
print(results)

pipeline_end = time.time()
print(f"Total pipeline runtime: {(pipeline_end - pipeline_start)/60:.2f} minutes")

print("CTAB-GAN+ (MAGIC) PIPELINE FINISHED SUCCESSFULLY")


ROOT: /content/drive/MyDrive/katabatic1

▶ Splitting MAGIC dataset
Loaded data with shape: (19020, 11)
Saved train/test full data
Train size: (15216, 11), Test size: (3804, 11)
Train label distribution:
 class
g    0.648396
h    0.351604
Name: proportion, dtype: float64
Test label distribution:
 class
g    0.648265
h    0.351735
Name: proportion, dtype: float64
Saved X/y split
Training shape: (15216, 10) (15216,)
Test shape: (3804, 10) (3804,)
✔ Split complete

▶ Training CTAB-GAN+ (MAGIC)
✔ Training complete in 2.34 minutes

Synthetic shapes:
X: (11412, 10)
y: (11412, 1)

Synthetic label distribution:
class
g        0.569576
h        0.430424
Name: proportion, dtype: float64

▶ Encoding data for TSTR
✔ Encoding complete

▶ Running TSTR evaluation

Results saved to: Results/magic/ctabgan_plus_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.5347
F1 Score: 0.5356
AUC: 0.4670

MLP:
Accuracy: 0.4317
F1 Score: 0.4413
AUC: 0.4484

RF:
Accuracy: 0.5271
F1 Score: 0.5376
AUC: 0.5050

XGBoos

In [ ]:
from pathlib import Path
import pandas as pd
import shutil

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "car.csv"
DATASET_DIR = ROOT / "sample_data" / "car"
SYNTH_DIR = ROOT / "synthetic" / "car" / "ctabgan_plus"

DATASET_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)

from katabatic.utils.split_dataset import split_dataset

print("Splitting CAR dataset")

split_dataset(
    input_csv=str(RAW_CSV),
    output_dir=str(DATASET_DIR),
    label_col="6",
    test_size=0.2,
    stratify=True,
    random_state=42
)

print("Split complete")

from katabatic.models.ctabgan_plus.adapter import CTABGANPlusAdapter

print("Training CTAB-GAN+")

model = CTABGANPlusAdapter()
model.train(
    dataset_dir=str(DATASET_DIR),
    synthetic_dir=str(SYNTH_DIR)
)

print("Training complete")

x_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")
y_synth = pd.read_csv(SYNTH_DIR / "y_synth.csv")

print("Synthetic shapes:")
print("X:", x_synth.shape)
print("y:", y_synth.shape)

print("Synthetic label distribution:")
print(y_synth.value_counts(normalize=True))

from sklearn.preprocessing import OrdinalEncoder

print("Encoding data for TSTR")

x_test = pd.read_csv(DATASET_DIR / "x_test.csv")
x_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")

encoder = OrdinalEncoder()

x_test_enc = encoder.fit_transform(x_test.astype(str))
x_synth_enc = encoder.transform(x_synth.astype(str))

pd.DataFrame(x_test_enc, columns=x_test.columns).to_csv(
    DATASET_DIR / "x_test_enc.csv", index=False
)
pd.DataFrame(x_synth_enc, columns=x_synth.columns).to_csv(
    SYNTH_DIR / "x_synth_enc.csv", index=False
)

shutil.move(DATASET_DIR / "x_test_enc.csv", DATASET_DIR / "x_test.csv")
shutil.move(SYNTH_DIR / "x_synth_enc.csv", SYNTH_DIR / "x_synth.csv")

print("Encoding complete")

from katabatic.evaluate.tstr.evaluation import TSTREvaluation

print("Running TSTR evaluation")

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(DATASET_DIR)
)

results = tstr.evaluate()

print("TSTR Results")
print(results)

print("CTAB-GAN+ PIPELINE FINISHED SUCCESSFULLY")


ROOT: /content/drive/MyDrive/katabatic1

▶ Splitting CAR dataset
Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
unacc    0.700434
acc      0.222142
good     0.039797
vgood    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
unacc    0.699422
acc      0.222543
good     0.040462
vgood    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)
✔ Split complete

▶ Training CTAB-GAN+
✔ Training complete

Synthetic shapes:
X: (1036, 6)
y: (1036, 1)

Synthetic label distribution:
6    
unacc    0.756757
acc      0.230695
vgood    0.011583
good     0.000965
Name: proportion, dtype: float64

▶ Encoding data for TSTR
✔ Encoding complete

▶ Running TSTR evaluation


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



Results saved to: Results/car/ctabgan_plus_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.6590
F1 Score: 0.5994

MLP:
Accuracy: 0.7572
F1 Score: 0.7254

RF:
Accuracy: 0.7457
F1 Score: 0.7138

XGBoost:
Accuracy: 0.7254
F1 Score: 0.6940

📊 TSTR RESULTS
{'LR': {'Accuracy': 0.6589595375722543, 'F1 Score': 0.5994497398781655}, 'MLP': {'Accuracy': 0.7572254335260116, 'F1 Score': 0.7253655524926581}, 'RF': {'Accuracy': 0.7456647398843931, 'F1 Score': 0.7138388303298199}, 'XGBoost': {'Accuracy': 0.7254335260115607, 'F1 Score': 0.6940496429785787}}

✅ CTAB-GAN+ PIPELINE FINISHED SUCCESSFULLY


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [07:10:25] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [4]:
pip install xgboost


In [ ]:
from pathlib import Path
import pandas as pd
import shutil
import time
import numpy as np

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "shuttle.csv"
DATASET_DIR = ROOT / "sample_data" / "shuttle"
SYNTH_DIR = ROOT / "synthetic" / "shuttle" / "ctabgan_plus"

DATASET_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
pipeline_start = time.time()

from katabatic.utils.split_dataset import split_dataset

print("Splitting SHUTTLE dataset")

split_dataset(
    input_csv=str(RAW_CSV),
    output_dir=str(DATASET_DIR),
    label_col="class",
    test_size=0.2,
    stratify=True,
    random_state=42
)

print("Split complete")

from katabatic.models.ctabgan_plus.adapter import CTABGANPlusAdapter

print("Training CTAB-GAN+ (SHUTTLE, 300 epochs)")

train_start = time.time()

model = CTABGANPlusAdapter()
model.train(
    dataset_dir=str(DATASET_DIR),
    synthetic_dir=str(SYNTH_DIR)
)

train_end = time.time()
print(f"Training complete in {(train_end - train_start)/60:.2f} minutes")

x_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")
y_synth = pd.read_csv(SYNTH_DIR / "y_synth.csv")

print("Synthetic shapes:")
print("X:", x_synth.shape)
print("y:", y_synth.shape)

print("Synthetic label distribution:")
print(y_synth.value_counts(normalize=True))

from sklearn.preprocessing import OrdinalEncoder

print("Encoding data for TSTR (unknown-safe)")

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

x_test = pd.read_csv(DATASET_DIR / "x_test.csv")
x_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")

x_test_enc = encoder.fit_transform(x_test.astype(str))
x_synth_enc = encoder.transform(x_synth.astype(str))

pd.DataFrame(x_test_enc, columns=x_test.columns).to_csv(
    DATASET_DIR / "x_test_enc.csv", index=False
)
pd.DataFrame(x_synth_enc, columns=x_synth.columns).to_csv(
    SYNTH_DIR / "x_synth_enc.csv", index=False
)

shutil.move(DATASET_DIR / "x_test_enc.csv", DATASET_DIR / "x_test.csv")
shutil.move(SYNTH_DIR / "x_synth_enc.csv", SYNTH_DIR / "x_synth.csv")

print("Encoding complete")

print("Reindexing labels for XGBoost compatibility")

y_train = pd.read_csv(DATASET_DIR / "y_train.csv")
y_test = pd.read_csv(DATASET_DIR / "y_test.csv")
y_synth = pd.read_csv(SYNTH_DIR / "y_synth.csv")

label_col = y_train.columns[0]

all_labels = pd.concat([y_train, y_test, y_synth])[label_col].unique()
all_labels = sorted(all_labels)

label_mapping = {old: new for new, old in enumerate(all_labels)}
print("Label mapping:", label_mapping)

y_train[label_col] = y_train[label_col].map(label_mapping)
y_test[label_col] = y_test[label_col].map(label_mapping)
y_synth[label_col] = y_synth[label_col].map(label_mapping)

y_train.to_csv(DATASET_DIR / "y_train.csv", index=False)
y_test.to_csv(DATASET_DIR / "y_test.csv", index=False)
y_synth.to_csv(SYNTH_DIR / "y_synth.csv", index=False)

print("Label reindexing complete")

from katabatic.evaluate.tstr.evaluation import TSTREvaluation

print("Running TSTR evaluation (LR, MLP, RF only)")

tstr_start = time.time()

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(DATASET_DIR),
    skip_models=["XGBoost"]
)

results = tstr.evaluate()

tstr_end = time.time()
print(f"TSTR completed in {(tstr_end - tstr_start)/60:.2f} minutes")

print("TSTR Results")
print(results)

pipeline_end = time.time()
print(f"Total pipeline runtime: {(pipeline_end - pipeline_start)/60:.2f} minutes")

print("CTAB-GAN+ (SHUTTLE, 300 epochs) PIPELINE FINISHED SUCCESSFULLY")


ROOT: /content/drive/MyDrive/katabatic1

▶ Splitting SHUTTLE dataset
Loaded data with shape: (58000, 10)
Saved train/test full data
Train size: (46400, 10), Test size: (11600, 10)
Train label distribution:
 class
1    0.785970
4    0.153491
5    0.056336
3    0.002953
2    0.000862
7    0.000216
6    0.000172
Name: proportion, dtype: float64
Test label distribution:
 class
1    0.785948
4    0.153534
5    0.056293
3    0.002931
2    0.000862
7    0.000259
6    0.000172
Name: proportion, dtype: float64
Saved X/y split
Training shape: (46400, 9) (46400,)
Test shape: (11600, 9) (11600,)
✔ Split complete

▶ Training CTAB-GAN+ (SHUTTLE, 300 epochs)
✔ Training complete in 14.35 minutes

Synthetic shapes:
X: (34800, 9)
y: (34800, 1)

Synthetic label distribution:
class
1        0.988420
4        0.009483
6        0.000690
5        0.000603
2        0.000402
7        0.000374
3        0.000029
Name: proportion, dtype: float64

▶ Encoding data for TSTR (unknown-safe)
✔ Encoding complete

▶ Rein

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [13:05:52] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/shuttle/ctabgan_plus_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7161
F1 Score: 0.6695

MLP:
Accuracy: 0.7390
F1 Score: 0.7165

RF:
Accuracy: 0.7322
F1 Score: 0.6987

XGBoost:
Accuracy: 0.7697
F1 Score: 0.7178
✔ TSTR completed in 0.44 minutes

📊 TSTR RESULTS
{'LR': {'Accuracy': 0.7161206896551724, 'F1 Score': 0.6695090713353511}, 'MLP': {'Accuracy': 0.7389655172413793, 'F1 Score': 0.716485157160349}, 'RF': {'Accuracy': 0.7322413793103448, 'F1 Score': 0.6986780335931357}, 'XGBoost': {'Accuracy': 0.7696551724137931, 'F1 Score': 0.7178311820675178}}

⏱️ Total pipeline runtime: 14.81 minutes

✅ CTAB-GAN+ (SHUTTLE, 300 epochs) PIPELINE FINISHED SUCCESSFULLY
